In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "marketplace_catalog"
SCHEMA = "team6_marketplace"

def bronze_table(name):
    return f"{CATALOG}.{SCHEMA}.bronze_{name}"

def silver_table(name):
    return f"{CATALOG}.{SCHEMA}.silver_{name}"

print("Silver layer configuration completed.")


Silver layer configuration completed.


In [0]:

customers = spark.table(bronze_table("customers"))

customers_clean = (
    customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_zip_code_prefix",
                F.col("customer_zip_code_prefix").cast("int"))
    .withColumn("customer_city", F.trim(F.col("customer_city")))
    .withColumn("customer_state", F.trim(F.col("customer_state")))
)

(
    customers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table("customers"))
)

print("Customers Silver table created:", customers_clean.count())


Customers Silver table created: 99441


In [0]:

orders = spark.table(bronze_table("orders"))

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean = orders.dropDuplicates(["order_id"])

for column in date_columns:
    orders_clean = orders_clean.withColumn(
        column,
        F.to_timestamp(F.col(column))
    )

(
    orders_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table("orders"))
)

print("Orders Silver table created:", orders_clean.count())


Orders Silver table created: 99441


In [0]:

items = spark.table(bronze_table("order_items"))

items_clean = (
    items
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .withColumn("shipping_limit_date",
                F.to_timestamp("shipping_limit_date"))
)

(
    items_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table("order_items"))
)

print("Order items Silver table created:", items_clean.count())


Order items Silver table created: 112650


In [0]:

sellers = spark.table(bronze_table("sellers"))

sellers_clean = (
    sellers
    .dropDuplicates(["seller_id"])
    .withColumn("seller_zip_code_prefix",
                F.col("seller_zip_code_prefix").cast("int"))
    .withColumn("seller_city", F.trim(F.col("seller_city")))
    .withColumn("seller_state", F.trim(F.col("seller_state")))
)

(
    sellers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table("sellers"))
)

print("Sellers Silver table created:", sellers_clean.count())


Sellers Silver table created: 3095


In [0]:

products = spark.table(bronze_table("products"))

numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_clean = (
    products
    .dropDuplicates(["product_id"])
    .withColumn("product_category_name",
                F.trim(F.col("product_category_name")))
)

for column in numeric_columns:
    products_clean = products_clean.withColumn(
        column,
        F.col(column).cast("int")
    )

(
    products_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table("products"))
)

print("Products Silver table created:", products_clean.count())


Products Silver table created: 32951
